# 어댑터 통제군 평가 — big7 dev

**질문**: 11차가 옛 데이터로 학습한 모델보다 낫다면, 그건 **데이터 때문인가**.

11차는 시험지(dev)를 바꿨으므로 옛 수치와 가로 비교가 안 된다. 그래서 **옛 데이터로
학습한 어댑터를 big7 dev에서 그대로 평가**해 같은 시험지 위의 통제군을 만든다.
재학습은 없다. 어댑터당 dev 1427개 생성(약 25분).

- 셀 1~3 · 6 · 7은 학습 노트북(`finetune_whisper_608_colab.ipynb`)의 셀을
  **그대로** 가져왔다. 셀 4는 LoRA를 새로 만드는 부분만 어댑터 로드로 바꿨고,
  셀 5의 전처리 함수는 글자 그대로 같다 — 학습 중 평가와 입력·지표가 같아야
  숫자가 이어진다.
- 판정 규칙은 대장 「11차 — 곡선은 나왔다」에 **숫자를 보기 전에** 적었다.
  이 노트북이 그 규칙을 코드로 옮겨 자동으로 판정한다(문턱을 여기서 바꾸지 마라).
- 마지막 셀의 **평가 경로 검증**이 먼저다. 이 노트북이 재현한 11차 dev 값이 학습 중
  기록(전체 0.1259 · 단어 0.1305 · 문장 0.1211)과 안 맞으면 그 아래 판정은 무효다.
- 어댑터별 결과는 드라이브에 저장한다. 세션이 끊겨도 마지막 셀만 다시 돌리면 이어진다.
  (끊긴 뒤에는 셀 1부터 다시 — 어댑터 로드와 데이터가 메모리에 있어야 한다.)

준비물: 드라이브에 `colab_trainset_big7.zip`과 `whisper-608-lora-11차`,
`whisper-608-lora-7차재현`(있으면 `-10차`도). 없는 어댑터는 건너뛴다.


## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (환경 핀). numpy가 바뀔 때만 재시작을 요구한다

In [ ]:
# ══ 환경 고정 ═══════════════════════════════════════════════
# 회차 간 비교를 하려면 프레임워크가 회차마다 바뀌면 안 된다. 7차(8/18)는
# transformers 4.x, 7차재현(8/20)은 5.15.1이었고 그 사이에 같은 배치 설정이
# T4에서 OOM으로 죽었다. 올릴 때는 회차를 새로 시작하고 대장에 환경을 적는다.
TFV = "5.15.1"

# numpy는 Colab 이미지가 든 버전을 그대로 고정한다. 숫자를 하드코딩하면
# 이미지가 바뀔 때마다 다시 깨진다:
#   너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy, subprocess, sys
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정 / transformers →", TFV)

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U "transformers=={TFV}" datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

# **새 프로세스로** 확인한다. 이 커널이 이미 들고 있는 모듈은 pip이 디스크를
# 바꿔도 안 바뀌므로, 커널 안에서 import해 확인하면 거짓 통과한다.
_out = subprocess.run(
    [sys.executable, "-c",
     "import numpy, transformers; print(numpy.__version__, transformers.__version__)"],
    capture_output=True, text=True)
print("디스크:", (_out.stdout or _out.stderr).strip())
_disk = _out.stdout.split()

if len(_disk) != 2 or _disk[1] != TFV:
    raise SystemExit(f"transformers가 {TFV}가 아니다. 위 pip 출력을 확인해라.")
# numpy만이 재시작을 강제한다 — 이 커널이 이미 로드했기 때문이다.
# transformers·datasets 등은 아직 import 전이라 새 파일이 그대로 읽힌다.
if _disk[0] != NPV:
    raise SystemExit(f"numpy가 {NPV} → {_disk[0]}로 바뀌었다."
                     " [런타임 → 세션 다시 시작] 후 셀 2.5부터 다시 실행해라.")
print("✅ 설치 완료 — 재시작 없이 셀 2.5로 계속 진행")


## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
if not transformers.__version__.startswith("5.15"):
    raise SystemExit("transformers 핀이 안 먹었다 — 셀 2를 다시 실행해라.")

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + 업로드한 zip 풀기 + 경로 확인
`ZIP`을 본인이 올린 위치로 수정.

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset_big7.zip"   # ← 업로드한 zip 경로
# big7 = VS01 병합본(11차부터). train 2281→10485, dev 326→1427, test 417→2673 —
# **train만 4.6배가 아니라 dev·test도 커졌다.** 그래서 옛 dev/test 수치(0.3101 등)를
# 그대로 인용하면 안 된다 — 시험지 자체가 다른 시험지다(아래 셀 8 참고).
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())

# dev는 선택이 아니다. dev 없이 돌리면 test로 체크포인트를 고르게 되고, 그러면
# 보고되는 CER이 홀드아웃 성능이 아니라 "시험지를 보며 고른 점수"가 된다.
# 4~6차 배치가 그 상태였다. 없으면 여기서 멈추고 재패키징을 요구한다.
if not os.path.exists(os.path.join(BASE, "dev.jsonl")):
    raise SystemExit(
        "dev.jsonl이 없다 — 이 zip은 dev 스플릿 없이 만들어졌다.\n"
        "prepare_colab_trainset.py를 --dev-speaker-frac 0.1 로 다시 돌려 패키징해라.\n"
        "(dev 없이 진행하면 test로 최고 체크포인트를 고르게 되어 수치를 못 쓴다)"
    )

for f in ["train.jsonl","dev.jsonl","test.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + 어댑터 로드 (원본 셀 4에서 LoRA 새로 만들기만 어댑터 로드로 교체)

In [ ]:
# 할당자가 크기별로 굳은 세그먼트를 재사용하지 못해 여유가 있는데도 OOM이 나는
# 경우가 있다(생성 평가는 배치마다 시퀀스 길이가 달라 특히 그렇다). 이 설정은
# **CUDA 컨텍스트가 만들어지기 전에** 걸려야 효과가 있어 맨 위에 둔다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

# 베이스 모델. zero-shot 비교(셀 10) 결과로 large-v3-turbo를 채택했다.
#
#   모델              단어 CER   문장 CER
#   small               2.000      0.404
#   medium              1.376      0.363
#   large-v3-turbo      0.890      0.321   ← 두 지표 모두 승
#
# 단어가 모델 크기에 훨씬 민감하다(2.2배 개선 vs 문장 1.26배). 앱 최약점이
# 단어 수준 구음장애라 이 차이가 그대로 제품 성능이다. turbo는 디코더가 얕아
# large-v3(1550M)보다 훨씬 빠르면서 T4에 들어간다.
#
# 되돌리려면 이 한 줄만 바꾼다. 프로세서도 같은 ID로 로드되므로 mel 빈 수
# (large-v3 계열 128 vs small/medium 80)가 자동으로 맞는다 — 프로세서를
# 재사용하면 입력이 조용히 깨진다.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# **fp32로 명시 로드해야 한다.** large-v3-turbo는 config에 torch_dtype: float16이
# 박혀 있어 최신 transformers가 fp16 가중치로 올린다. 그러면 특징 추출기가 내놓는
# fp32 입력과 어긋나 인코더 conv에서 터진다:
#   RuntimeError: Input type (float) and bias type (c10::Half) should be the same
# small은 config가 fp32라 이 문제가 없어 지금까지 안 드러났다.
#
# 혼합정밀은 Trainer의 fp16=True(autocast)가 맡는다. 가중치까지 fp16으로 두고
# 학습하면 마스터 가중치가 없어 불안정하다. T4는 Turing이라 bf16이 없어 이 조합이
# 유일한 선택지다.
# **attn_implementation을 명시한다.** 기본값에 맡기면 transformers 버전에 따라
# eager로 떨어질 수 있고, 그러면 어텐션 확률 행렬이 통째로 저장된다:
#   배치4 x 헤드20 x 1500 x 1500 x 2B = 360MB/층  x  인코더 32층 = 11.5GB
# sdpa는 그걸 아예 만들지 않는다. 이 한 줄이 T4에 들어가느냐 마느냐를 가른다.
# (셀 2가 transformers를 핀 없이 -U로 깔기 때문에 기본값에 기댈 수 없다.)
try:
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, attn_implementation='sdpa')
except (ValueError, ImportError) as e:
    print('sdpa 거부됨:', e)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
print('어텐션 구현:', getattr(model.config, '_attn_implementation', '?'),  # 'sdpa'여야 한다
      '| torch', torch.__version__)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# ══ 어댑터 로드 — 학습된 LoRA를 붙인다(새로 만들지 않는다) ═══════════════
# 원본 셀 4는 여기서 get_peft_model(LoraConfig(...))로 **새** LoRA를 만든다.
# 통제군 평가는 학습을 안 하므로 저장된 어댑터를 읽는다. 어댑터 여러 개를 한
# 모델에 얹어 set_adapter로 바꿔 가며 잰다(베이스 3.2GB를 한 번만 올린다).
from peft import PeftModel

# 평가 순서. 11차를 먼저 재는 이유: 학습 중 기록된 dev 값과 이 노트북의 값이
# 맞는지(=평가 경로가 같은지) 가장 먼저 확인해야 나머지 숫자를 믿을 수 있다.
ADAPTERS = ["11차", "7차재현", "10차"]
ADAPTER_DIR = "/content/drive/MyDrive/whisper-608-lora-{}"

found, absent = [], []
for _n in ADAPTERS:
    (found if os.path.isdir(ADAPTER_DIR.format(_n)) else absent).append(_n)
print("어댑터 있음:", found)
if absent:
    print("어댑터 없음:", absent, "→ 이 회차는 건너뛴다(재학습 없이는 통제군이 못 된다)")
if not found:
    raise SystemExit("드라이브에 어댑터가 하나도 없다 — whisper-608-lora-<회차> 폴더를 확인해라.")

model = PeftModel.from_pretrained(model, ADAPTER_DIR.format(found[0]), adapter_name=found[0])
for _n in found[1:]:
    model.load_adapter(ADAPTER_DIR.format(_n), adapter_name=_n)
model.eval()
print("로드 완료. 활성:", model.active_adapter)

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제

# ── 평가 시간의 대부분은 여기서 결정된다 ─────────────────────────
# 자기회귀 생성이라 **배치 안에서 가장 긴 시퀀스**가 그 배치의 시간을 정한다.
# 그리고 학습 전/초기 모델은 환각으로 거의 항상 상한까지 뱉는다 — 즉 상한값이
# 곧 평가 비용이다.
#
# 실측 라벨 길이 (colab_trainset_big6):
#     train 단어  중앙 1자 · 최대  2자
#     train 문장  중앙 13자 · p95 34자 · 최대 93자
#     dev  문장  최대 56자   |   test 문장 최대 44자
#
# 100토큰은 dev/test 최장(56자)의 두 배 가까이라 낭비다. 72로 낮춰 환각 꼬리를
# 자른다. 잘림 위험은 없다 — 한국어는 음절당 1토큰 안팎이라 56자면 60토큰 미만이다.
model.generation_config.max_new_tokens = 72

# **KV 캐시를 반드시 켠다.** 위에서 config.use_cache=False로 둔 건 학습용 관례인데,
# 그게 생성까지 따라가면 매 토큰마다 전체 시퀀스를 재계산해 평가가 몇 배 느려진다.
# (gradient_checkpointing을 안 쓰므로 학습에도 use_cache=False가 필요 없지만,
#  건드리면 학습 조건이 바뀌므로 생성 쪽만 명시적으로 켠다.)
model.generation_config.use_cache = True


## 셀 5a — 데이터 (dev만)

In [ ]:
import collections
from datasets import load_dataset, Audio

# 통제군 평가는 dev만 본다. train 10485개의 mel을 또 뽑을 이유가 없다(수십 분).
# 화자·과제 라벨은 prep이 컬럼을 지우기 전에 원본 jsonl에서 뽑는다.
# audio 필드가 "wav/<화자>/<파일>.wav"라 폴더명이 화자 코드다.
raw_dev = [json.loads(l) for l in open(f"{BASE}/dev.jsonl", encoding="utf-8")]
dev_speakers = [os.path.basename(os.path.dirname(r["audio"])) for r in raw_dev]
dev_task_types = [r["task_type"] for r in raw_dev]
test_task_types = None   # 셀 7(지표)이 이 이름을 본다. 여기선 test를 안 잰다

ds = load_dataset("json", data_files={"dev": f"{BASE}/dev.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))


def prep(b):   # 원본 셀 5와 같은 함수. 바꾸면 학습 중 평가와 입력이 달라진다
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b


ds = ds.map(prep, remove_columns=ds["dev"].column_names, writer_batch_size=20)

assert len(ds["dev"]) == len(dev_speakers) == len(dev_task_types), "행 수가 안 맞는다"
print(ds)
print("화자", len(set(dev_speakers)), "명 ·", dict(collections.Counter(dev_task_types)))


## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 (전체 + 단어/문장 분리)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")

# 에폭 평가에서 **단어/문장 CER을 따로** 낸다. 9차의 판정 지표가 dev 단어 CER이라
# 전체 CER만으로는 판단할 수 없다(단어가 세그먼트의 절반이지만 짧아서 전체
# CER에 묻힌다).
#
# 어느 스플릿인지는 **길이로** 가린다 — dev 326 vs test 417이라 겹치지 않는다.
# 재패키징으로 두 길이가 같아지면 이 분기가 조용히 틀리므로 아래에서 막는다.
TASK_TYPES = {}
for _name, _tt in (("dev", dev_task_types), ("test", test_task_types)):
    if not _tt:
        continue
    if len(_tt) in TASK_TYPES:
        raise SystemExit(f"dev와 test 길이가 같다({len(_tt)}) — 길이로 스플릿을 못 가른다.")
    TASK_TYPES[len(_tt)] = _tt

def compute_metrics(p):
    lab = p.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    pred = processor.tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    ref  = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    out = {"cer": cer_metric.compute(predictions=pred, references=ref)}
    tt = TASK_TYPES.get(len(ref))
    if tt:
        for _short, _key in (("word", "wordlist"), ("sent", "narrative")):
            idx = [i for i, t in enumerate(tt) if t == _key]
            if idx:
                out[f"cer_{_short}"] = cer_metric.compute(
                    predictions=[pred[i] for i in idx],
                    references=[ref[i] for i in idx])
    return out

## 셀 5 — 어댑터별 dev 평가 + 판정

In [ ]:
import gc, json, re, time, unicodedata, warnings
import transformers as _tf
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
_tf.logging.set_verbosity_error(); warnings.filterwarnings("ignore")   # 출력 폭탄 억제(학습 노트북 셀 8과 같다)

# big6 dev 화자 5명. big7 dev는 이들 + 신규 8명이다. 같은 화자라도 VS01 클립은
# 새 데이터지만 **화자는 어느 통제군도 학습에서 본 적이 없다**(스플릿이 화자 단위).
OLD_DEV_SPEAKERS = {"BHL", "KDS", "KJS", "LTY", "YDE"}   # 로컬 big6 dev.jsonl에서 뽑았다(5명)
RESULT_PATH = "/content/drive/MyDrive/whisper-608-control-big7-dev.json"
PRED_PATH   = "/content/drive/MyDrive/whisper-608-control-big7-preds-{}.json"

# 학습 중 기록된 11차 dev 값(대장 11차 절). 이 노트북이 같은 값을 재현해야 평가
# 경로가 학습 중 평가와 같다는 뜻이다. 어긋나면 이후 비교 전부를 유보한다.
LOGGED_11 = {"전체": 0.1259, "단어": 0.1305, "문장": 0.1211}
SANITY_TOL = 0.002

# ── 사전 등록한 판정 규칙(대장 「11차 — 곡선은 나왔다」). 숫자를 보기 전에 정했다 ──
WORD_GAIN_MIN = 0.02    # 단어 CER이 통제군보다 이만큼 이상 낮아야 데이터 효과
SENT_COST_MAX = 0.01    # 문장 CER이 통제군보다 이만큼 넘게 나쁘면 대가가 크다


def _norm(t):
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"[^\w가-힣\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def summarize(hyps, refs, speakers, task_types, cer_fn):
    """전체·단어·문장, 그리고 단어를 옛/신규 화자로 갈라 CER과 정확도를 낸다."""
    n = len(refs)
    old = [s in OLD_DEV_SPEAKERS for s in speakers]
    groups = {
        "전체": list(range(n)),
        "단어": [i for i in range(n) if task_types[i] == "wordlist"],
        "문장": [i for i in range(n) if task_types[i] == "narrative"],
        "단어·옛화자": [i for i in range(n) if task_types[i] == "wordlist" and old[i]],
        "단어·신규화자": [i for i in range(n) if task_types[i] == "wordlist" and not old[i]],
    }
    out = {}
    for g, idx in groups.items():
        if not idx:
            continue
        out[g] = {"n": len(idx),
                  "cer": cer_fn([hyps[i] for i in idx], [refs[i] for i in idx])}
        if g.startswith("단어"):
            out[g]["acc"] = sum(_norm(hyps[i]) == _norm(refs[i]) for i in idx) / len(idx)
    return out


def verdict(new, ctl):
    """사전 등록 규칙을 그대로 옮긴 판정. new/ctl은 summarize 결과."""
    gain = ctl["단어"]["cer"] - new["단어"]["cer"]        # 양수 = 11차가 좋다
    cost = new["문장"]["cer"] - ctl["문장"]["cer"]        # 양수 = 11차가 나쁘다
    if gain >= WORD_GAIN_MIN and cost <= SENT_COST_MAX:
        v = "데이터 효과 있음"
    elif gain >= WORD_GAIN_MIN:
        v = "단어는 이기지만 문장 대가가 크다 (트레이드오프 — 보류)"
    elif gain > -WORD_GAIN_MIN:
        v = "차이 없음 (단어 차이가 문턱 0.02 미만)"
    else:
        v = "통제군이 더 좋다 (데이터를 늘렸는데 단어가 나빠졌다)"
    return v, gain, cost


args = Seq2SeqTrainingArguments(
    output_dir="/content/ctl_tmp",
    per_device_eval_batch_size=8,      # 학습 중 평가와 같은 값(16에서 OOM 났다 — 셀 8 주석)
    dataloader_num_workers=2, fp16=True,
    predict_with_generate=True, remove_unused_columns=False,
    label_names=["labels"], report_to="none")

results = json.load(open(RESULT_PATH, encoding="utf-8")) if os.path.exists(RESULT_PATH) else {}
for name in found:
    if name in results:
        print(f"[{name}] 이미 잰 결과가 있다 — 건너뜀")
        continue
    model.set_adapter(name)
    assert model.active_adapter == name, f"어댑터가 안 바뀌었다: {model.active_adapter}"
    trainer = Seq2SeqTrainer(model=model, args=args, data_collator=collator,
                             compute_metrics=compute_metrics, processing_class=processor)
    t0 = time.time()
    pred = trainer.predict(ds["dev"])
    lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyps = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    res = summarize(hyps, refs, dev_speakers, dev_task_types,
                    lambda h, r: cer_metric.compute(predictions=h, references=r))
    results[name] = res
    json.dump(results, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump([{"spk": s, "task": t, "ref": r, "hyp": h}
               for s, t, r, h in zip(dev_speakers, dev_task_types, refs, hyps)],
              open(PRED_PATH.format(name), "w", encoding="utf-8"), ensure_ascii=False)
    print(f"[{name}] {(time.time()-t0)/60:.1f}분 · 전체 {res['전체']['cer']:.4f} · "
          f"단어 {res['단어']['cer']:.4f} · 문장 {res['문장']['cer']:.4f}", flush=True)
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

# ══ 요약 ═════════════════════════════════════════════════════
cols = ["전체", "단어", "문장", "단어·옛화자", "단어·신규화자"]
first = results[next(iter(results))]
print()
print("  " + "어댑터".ljust(10) + "".join(c.rjust(14) for c in cols))
for name, r in results.items():
    print("  " + name.ljust(10) + "".join((f"{r[c]['cer']:.4f}" if c in r else "-").rjust(14) for c in cols))
print("  " + "n".ljust(10) + "".join(str(first[c]["n"]).rjust(14) for c in cols))

if "11차" in results:
    print()
    diff = {k: abs(results["11차"][k]["cer"] - v) for k, v in LOGGED_11.items()}
    ok = all(x <= SANITY_TOL for x in diff.values())
    print("[평가 경로 검증] 학습 중 기록 대비 차이:", {k: round(x, 4) for k, x in diff.items()},
          "→", "일치" if ok else "**어긋난다 — 아래 판정을 믿지 마라**")
    for ctl in ("7차재현", "10차"):
        if ctl in results:
            v, gain, cost = verdict(results["11차"], results[ctl])
            print(f"[판정 vs {ctl}] 단어 이득 {gain:+.4f}(문턱 {WORD_GAIN_MIN}) · "
                  f"문장 대가 {cost:+.4f}(문턱 {SENT_COST_MAX}) → {v}")
            for g in ("단어·옛화자", "단어·신규화자"):
                if g in results[ctl] and g in results["11차"]:
                    c0, c1 = results[ctl][g]["cer"], results["11차"][g]["cer"]
                    print(f"      {g}: 통제 {c0:.4f} → 11차 {c1:.4f}  (이득 {c0 - c1:+.4f}, n={results['11차'][g]['n']})")


## 셀 6 — 어휘 분해 (마지막 셀을 돌린 뒤)

저장된 예측만 읽는다. 재평가 없음, 몇 초.


In [ ]:
# ══ 어휘 분해 — 통제군이 학습에서 본 적 있는 단어와 없는 단어를 갈라 본다 ══
# 저장해 둔 예측(whisper-608-control-big7-preds-*.json)만 읽는다. 재평가가 없다.
#
# 왜 보나: 608 단어는 닫힌 어휘(dev 단어의 99.8%가 big7 학습 어휘 안)라서,
# 11차의 단어 이득에는 "이 단어들을 33번씩 봤다"가 섞여 있다. 통제군(big6)은
# 학습 어휘가 203종이라 dev 단어의 9.4%가 **통제군에게는 처음 보는 단어**다.
# 그 부분에서 통제군이 얼마나 나쁜지가 "어휘 밖 단어에서 파인튜닝 모델이 어떻게
# 되는가"의 (약한) 단서다. 앱 표적 단어의 85.6%가 608 어휘 밖이다(대장 참고).
import json, os

BIG6_VOCAB = set("""가 가위 감 강 개 거울 걸 겁 겉 곰 공 과일 구 국 군 권투 귀 귤 그림 극장 글 금 기차 김 까 껌 꽃 꿀 꿈 나 나무 날개 남 냄새 네 노래 녹 논 논밭 눈 느낌 능 다 달 달걀 당근 덕 도 돈 돌 동생 등대 따 딸기 땅콩 떡 똥 뜰 라 마 마음 만 말 모자 목표 몸 무 문 문제 물 밀 바 바다 발 밤 밥 방 방석 배 뱀 벌 별 병 병원 북 불 붓 빗 빠 빵 뼈 뿔 사 사과 사람 산 삼 상 새 성 소 손 송곳 시 시간 신발 실 쌀 아 아들 안개 안경 알 앞 야 약 약국 양 어 에 여 연 연필 영 예 오 옷 와 왕 왜 요 욕심 용 우 우산 웨 위 유 육 육군 으 은 이 일 입 잎 자 자연 잔 장갑 잼 저녁 전화 절 점 접시 조 종 죄 쥐 집 짜 차 참새 창 창문 초 총 추 춤 칠 침 칼 컵 코 콩 키 타 탈 탑 터 턱 털 토끼 톱 파 팔 팥죽 편지 폐 풀 풍선 피 하 학교 해 허리 혀 형 호박 활 회사 흙""".split())   # big6 학습 단어 203종
PRED = "/content/drive/MyDrive/whisper-608-control-big7-preds-{}.json"
RES = json.load(open("/content/drive/MyDrive/whisper-608-control-big7-dev.json", encoding="utf-8"))


def _lev(a, b):
    p = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        c = [i]
        for j, y in enumerate(b, 1):
            c.append(min(p[j] + 1, c[j - 1] + 1, p[j - 1] + (x != y)))
        p = c
    return p[-1]


def _cer(rows):
    return sum(_lev(r["hyp"], r["ref"]) for r in rows) / max(1, sum(len(r["ref"]) for r in rows))


print("어댑터".ljust(10) + "단어 전체".rjust(12) + "통제어휘 안".rjust(14) + "통제어휘 밖".rjust(14) + "   (n 전체/안/밖)")
show = {}
for name in ("11차", "7차재현", "10차"):
    path = PRED.format(name)
    if not os.path.exists(path):
        print(name.ljust(10) + "예측 파일 없음 — 건너뜀")
        continue
    rows = [r for r in json.load(open(path, encoding="utf-8")) if r["task"] == "wordlist"]
    inv = [r for r in rows if r["ref"].strip() in BIG6_VOCAB]
    oov = [r for r in rows if r["ref"].strip() not in BIG6_VOCAB]
    # 계산 방식 검증: 앞 셀이 낸 단어 CER과 같아야 이 표를 믿을 수 있다
    diff = abs(_cer(rows) - RES[name]["단어"]["cer"])
    flag = "" if diff < 5e-4 else f"   경고: 앞 셀 값과 {diff:.4f} 어긋남 — 이 행을 믿지 마라"
    print(name.ljust(10) + f"{_cer(rows):12.4f}{_cer(inv):14.4f}{_cer(oov):14.4f}   ({len(rows)} / {len(inv)} / {len(oov)}){flag}")
    show[name] = oov

# 통제군이 처음 보는 단어를 어떻게 틀리는지 — 학습 어휘 안의 비슷한 단어로 끌려가는지 본다
for name in ("7차재현", "10차"):
    if name in show:
        wrong = [r for r in show[name] if r["hyp"].strip() != r["ref"].strip()]
        print()
        print(f"[{name}] 어휘 밖 단어 {len(show[name])}개 중 틀린 것 {len(wrong)}개, 예:")
        for r in wrong[:10]:
            tag = "(학습 어휘 안의 단어로 인식)" if r["hyp"].strip() in BIG6_VOCAB else ""
            print(f"   정답 {str(r['ref']):6} -> 인식 {str(r['hyp']):8} {tag}")
